# From a notebook to a panel other people can open

A notebook is a thing **you run**. A panel is a thing **other people open**. This notebook
is about the handoff between the two, and about the one question that handoff normally
leaves unanswered: *is the page I just built going to be cheap to interact with?*

The shape of what follows:

| step | where it happens |
|---|---|
| load and shape the data | here, in Python, however you like |
| declare the panel | here, with `Panel` — a thin mirror of dagpane's TOML |
| write it out | a directory: one CSV per source, one `app.toml` |
| **prove an interaction is cheap** | here, with `.explain()`, before anyone opens a browser |
| serve it | one static binary reading that directory |

After the fourth row the kernel is not in the loop. You can restart it, delete the venv,
or hand the directory to someone who has never installed Python, and the panel still
serves — because what serves it is a binary reading files.

**Prerequisite:** a `dagpane` binary. Either `cargo install --path crates/cli`,
or `cargo build --release -p dagpane-cli` in this checkout, or set `DAGPANE` to a path.

In [ ]:
import sys

sys.path.insert(0, '.')  # so this notebook finds dagpane_notebook.py beside it

from dagpane_notebook import Panel, find_dagpane

print('using', find_dagpane())

## 1. The analysis, in Python

This is your half, and dagpane has no opinion about it. Use pandas, polars, a warehouse
query, a REST call — anything that ends in rows. The cell below uses pandas when it is
installed and falls back to the standard library when it is not, so the notebook runs
either way.

What matters for the panel is the *shape* of what comes out: the columns you want to
filter on, group by, and show. Do the expensive, one-off work here — the joins, the
cleaning, the model scoring. dagpane is for the part that has to be fast **per
interaction**, not the part that runs once.

In [ ]:
import random

rng = random.Random(11)
CHANNELS = ('web', 'app', 'partner', 'referral')
DEVICES = ('desktop', 'mobile', 'tablet')

records = [
    {
        'day': f'2026-08-{d:02d}',
        'day_index': d,
        'channel': channel,
        'device': device,
        'signups': rng.randint(4, 380),
        'revenue': round(rng.uniform(40.0, 9_400.0), 2),
    }
    for d in range(1, 29)
    for channel in CHANNELS
    for device in DEVICES
]

try:
    import pandas as pd
    data = pd.DataFrame.from_records(records)
    # ...your real work goes here: joins, cleaning, feature columns, model scores.
    print(data.head())
except ImportError:
    data = records  # a list of dicts works just as well
    print(f'{len(data)} rows, no pandas needed')

## 2. Declare the panel

`Panel` maps one-to-one onto the manifest: `source`, `select`/`slider`/`checkbox`,
`cell(...).filter(...).group_by(...)`, `metric`/`bar`/`table`/`line`. That is deliberate —
the TOML it writes is one you could have typed, so this teaches the authoring surface
instead of hiding it, and you can delete the builder later and keep the file.

Two things in the cell below are the whole reason to build a panel on a graph rather than
a rerun, and they are worth reading closely:

* **`skip_when='all'`** — the dropdown's "all" option is not a conditional. It is a step
  that does nothing, and a step that does nothing leaves the cell holding the value it
  already had, so nothing below it runs. Note it is written with `param=`, never with a
  literal: `skip_when` is compared against the value the filter *resolves to*, so a literal
  can never match it and the control would silently be wired to nothing. The builder
  refuses that combination rather than letting you find out in a meeting.
* **`lifetime_revenue` reads `signups`, not `scoped`** — so no control on this page is
  upstream of it, and no control on this page can make it recompute. That is how you fence
  off the number that must not move from the ones that should.

In [ ]:
panel = Panel('Acquisition', subtitle='28 days, four channels — built in a notebook')

panel.source('signups', data)

panel.select('channel', ['all', *CHANNELS], label='Channel')
panel.slider('min_signups', 0, 380, step=20, label='At least this many signups')

(panel.cell('scoped', 'signups')
      .filter('signups', 'ge', param='min_signups')
      .filter('channel', 'eq', param='channel', skip_when='all'))

(panel.cell('by_channel', 'scoped')
      .group_by(by=['channel'], agg=[
          {'column': 'signups', 'agg': 'sum', 'as': 'signups'},
          {'column': 'revenue', 'agg': 'sum', 'as': 'revenue'},
      ])
      .sort('signups', descending=True))

(panel.cell('per_day', 'scoped')
      .group_by(by=['day_index'], agg=[{'column': 'signups', 'agg': 'sum', 'as': 'signups'}])
      .sort('day_index'))

(panel.cell('total_signups', 'scoped')
      .group_by(agg=[{'column': 'signups', 'agg': 'sum', 'as': 'total'}])
      .scalar('total'))

# The insensitive cell: the SET of devices present. Move the slider and the totals change,
# but the same three devices are still there — so this recomputes, produces the value it
# already had, and everything below it is left alone.
(panel.cell('devices_present', 'scoped')
      .group_by(by=['device'], agg=[{'agg': 'count', 'as': 'n'}])
      .select(['device'])
      .sort('device'))
panel.cell('device_count', 'devices_present').count()

# Reads the SOURCE. Fenced off from every control above.
(panel.cell('lifetime_revenue', 'signups')
      .group_by(agg=[{'column': 'revenue', 'agg': 'sum', 'as': 'total'}])
      .scalar('total'))

panel.metric('total_signups', 'Signups in view')
panel.metric('device_count', 'Devices')
panel.metric('lifetime_revenue', 'Revenue, all channels', prefix='$', decimals=0)
panel.bar('by_channel', 'channel', 'signups', title='Signups by channel')
panel.line('per_day', 'day_index', 'signups', title='Signups per day')
panel.table('by_channel', title='Detail', max_rows=10, id='channel_table')

print(panel.to_toml()[:700], '...')

## 3. Write it out, and compile it

`write()` produces the deliverable: a directory holding `app.toml` and one CSV per source.
A manifest's paths resolve against its own directory, so that directory is self-contained —
mount it, upload it, commit it, hand it over.

`check()` compiles the app and reports everything wrong with it **before** anything runs:
a misspelt column, a filter naming an input that does not exist, a cycle. A misspelt edge
is not quietly dropped, which is what would otherwise leave you with a control that looks
fine and never updates anything.

In [ ]:
panel.write('build/acquisition')
panel.check()

## 4. Look at the graph you actually built

The left column is height. A pass evaluates in ascending height, which is what makes it
impossible for a cell to see a mix of old and new inputs — the diamond below (`by_channel`,
`per_day`, `total_signups` and `devices_present` all reading `scoped`) resolves exactly once,
never with one parent updated and another stale.

This is printed from the declared edges, so it is available before the app has ever run.

In [ ]:
panel.graph()

## 5. The part a notebook can do that a browser cannot

`explain()` runs one interaction and returns **what it cost**, as a dict. Not a timing —
a count of which cells ran, which served the value they already had, which were never
looked at, and which panes went on the wire.

That turns "will this page be cheap to click around in" into an assertion you write beside
the analysis that produced it. Put these in the notebook, or lift them into CI: the same
`--set` flags work from a shell, and `--json` gives you the same structure.

In [ ]:
cost = panel.explain(channel='web')

print('\nnever looked at:', cost['untouched'])
print('panes sent:     ', cost['panes_sent'])

# The fenced-off cell is genuinely untouched — not recomputed-and-discarded.
assert 'lifetime_revenue' in cost['untouched'], 'the lifetime number is downstream of a control'

# And the page as a whole bought something by being a graph.
assert len(cost['panes_sent']) < cost['panes_total'], 'every pane repainted'
print('\nok')

Now watch the *other* claim — the one an annotation-based runtime cannot make. Moving the
slider changes the totals, so `by_channel` and `total_signups` recompute. But the set of
devices present does not change, so `devices_present` produces the value it already held and
`device_count` below it **does not run at all**.

Nobody declared that. It falls out of comparing the value the cell produced against the one
it had.

In [ ]:
cost = panel.explain(min_signups=120)

outcomes = {s['cell']: s['outcome'] for s in cost['trace']['steps']}
for cell in ('scoped', 'by_channel', 'devices_present', 'device_count'):
    print(f'  {cell:<18} {outcomes.get(cell, "never looked at")}')

assert outcomes.get('device_count') == 'reused', 'the short-circuit did not fire'
print('\nok — the pass stopped at devices_present')

## 6. Serve it

`serve()` blocks; interrupt the cell to stop it. Pass `background=True` to get a `Popen`
back and keep working — just remember a kernel restart orphans the process.

The page shows the same counts live, above the panes, and a repainted pane flashes its
border, so the thing you just asserted on is visible while you click.

> **There is no authentication in this version.** The default bind is loopback. Binding
> anything else makes the page reachable by anyone who can route to it, so put something
> that authenticates in front of it first. `SECURITY.md` covers this in full.

In [ ]:
# panel.serve()                       # blocks; interrupt to stop
# server = panel.serve(background=True)  # ...or keep working, then server.terminate()

## 7. What you have now

```
build/acquisition/
  app.toml        the app
  signups.csv     the data
```

Two files, and a binary that reads them. Nothing in that directory needs Python, a package
manager, or this notebook — which is the point. The same directory is what
`../bucket/serve.sh` refreshes from object storage on a schedule, and what the container
mounts:

```sh
dagpane run build/acquisition/app.toml
docker run --rm -p 8787:8787 -v "$PWD/build/acquisition:/app:ro" \
  mancube/dagpane:0.1.1 run /app/app.toml --host 0.0.0.0
```

**Where to go next**

* `../README.md` — fifteen worked apps by role, and the honest list of when *not* to reach
  for this.
* `../bucket/README.md` — the same panel, with its data coming from a cloud bucket on a
  refresh loop.
* Open `build/acquisition/app.toml` and edit it. That file is the app now; this notebook was
  scaffolding.